# **Semana 9: Pipelines - Procesamiento Batch con Datos Reales (NB2 - Práctico/Ejercicios)**

## **Propósito de la Sesión**
Aplicar procesamiento batch sobre un dataset real, implementando un pipeline completo de extracción, transformación y carga (ETL) que simule un entorno de producción. Además, aprenderemos a medir y comparar tiempos de ejecución para comprender la importancia de la eficiencia en pipelines de datos.

### **Objetivos de Aprendizaje**
Al finalizar este notebook, el estudiante será capaz de:
1. **Trabajar** con un dataset real de gran tamaño (ventas, reseñas, o similar).
2. **Diseñar** un pipeline batch que procese los datos por lotes.
3. **Implementar** transformaciones significativas (limpieza, agregaciones, uniones).
4. **Medir** y comparar tiempos de ejecución de diferentes enfoques.
5. **Analizar** el rendimiento y extraer conclusiones sobre optimización.

## **Configuración Inicial**

Importamos las librerías necesarias y preparamos el entorno para trabajar con datasets reales.

In [ ]:
# Instalación de librerías necesarias
!pip install pandas numpy matplotlib seaborn --quiet

# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import sqlite3
import requests
import io
import zipfile
from datetime import datetime

# Configuración de visualización
sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

print("Librerías importadas correctamente.")
print(f"Versión de pandas: {pd.__version__}")
print(f"Versión de numpy: {np.__version__}")

---
## **Parte 1: Obtención del Dataset Real**

Para este notebook, utilizaremos el dataset **"Online Retail"** de UCI Machine Learning Repository, que contiene transacciones de una tienda minorista online del Reino Unido entre 2010 y 2011. Es un dataset ideal para prácticas de procesamiento batch por su tamaño moderado y riqueza de datos.

**Descripción del dataset:**
*   **InvoiceNo**: Número de factura
*   **StockCode**: Código del producto
*   **Description**: Descripción del producto
*   **Quantity**: Cantidad vendida
*   **InvoiceDate**: Fecha de factura
*   **UnitPrice**: Precio unitario
*   **CustomerID**: ID del cliente
*   **Country**: País del cliente

In [ ]:
# URL del dataset (fuente oficial de UCI)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"

print("Descargando dataset 'Online Retail'...")
print("(Esto puede tomar unos segundos, el archivo tiene ~45MB)")

# Medir tiempo de descarga
start_time = time.time()

# Descargar el archivo Excel
try:
    response = requests.get(url)
    response.raise_for_status()  # Verificar que la descarga fue exitosa
    
    # Guardar el contenido en un archivo temporal
    with open('online_retail.xlsx', 'wb') as f:
        f.write(response.content)
    
    download_time = time.time() - start_time
    print(f"✅ Descarga completada en {download_time:.2f} segundos.")
    
    # Cargar el dataset con pandas
    df_raw = pd.read_excel('online_retail.xlsx')
    
    print(f"\nDataset cargado correctamente.")
    print(f"Dimensiones: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas")
    
except Exception as e:
    print(f"❌ Error en la descarga: {e}")
    print("Usaremos un dataset alternativo generado sintéticamente para continuar.")
    
    # Generar datos sintéticos como fallback
    np.random.seed(42)
    n_rows = 500000
    df_raw = pd.DataFrame({
        'InvoiceNo': np.random.choice(['A'+str(i) for i in range(1000, 9999)], n_rows),
        'StockCode': np.random.choice(['P'+str(i).zfill(5) for i in range(1, 500)], n_rows),
        'Description': np.random.choice(['Producto A', 'Producto B', 'Producto C', 'Producto D'], n_rows),
        'Quantity': np.random.randint(1, 50, n_rows),
        'InvoiceDate': pd.date_range(start='2023-01-01', periods=n_rows, freq='T'),
        'UnitPrice': np.round(np.random.uniform(1, 100, n_rows), 2),
        'CustomerID': np.random.choice(range(10000, 99999), n_rows),
        'Country': np.random.choice(['UK', 'France', 'Germany', 'Spain', 'Italy'], n_rows)
    })
    print(f"Dataset sintético generado: {df_raw.shape}")

# Mostrar información básica del dataset
print("\n--- Información del Dataset ---")
print(df_raw.info())

print("\n--- Primeras 5 filas ---")
display(df_raw.head())

---
## **Parte 2: Exploración y Limpieza Inicial**

Antes de procesar, necesitamos entender y limpiar los datos. Esto es parte esencial de cualquier pipeline batch.

In [ ]:
print("--- EXPLORACIÓN Y LIMPIEZA INICIAL ---")

# 1. Verificar valores nulos
print("\n1. Valores nulos por columna:")
null_counts = df_raw.isnull().sum()
print(null_counts[null_counts > 0])

# 2. Estadísticas descriptivas
print("\n2. Estadísticas descriptivas:")
display(df_raw[['Quantity', 'UnitPrice']].describe())

# 3. Identificar anomalías (cantidades negativas, precios cero)
print("\n3. Anomalías detectadas:")
print(f"  - Cantidades negativas: {(df_raw['Quantity'] < 0).sum()} registros")
print(f"  - Precios cero o negativos: {(df_raw['UnitPrice'] <= 0).sum()} registros")

# 4. Limpieza básica
df_clean = df_raw.copy()

# Eliminar filas con CustomerID nulo (no se puede asociar la venta a un cliente)
if 'CustomerID' in df_clean.columns:
    df_clean = df_clean.dropna(subset=['CustomerID'])
    print(f"\n4. Filas después de eliminar CustomerID nulos: {len(df_clean)}")

# Filtrar cantidades positivas y precios positivos (transacciones válidas)
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]
print(f"   Filas después de filtrar cantidades y precios positivos: {len(df_clean)}")

# Crear columna de monto total
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']

print("\nDataset limpio:")
print(f"Dimensiones finales: {df_clean.shape}")
display(df_clean.head())

---
## **Parte 3: Procesamiento Batch - Pipeline Completo**

Implementaremos un pipeline batch que realice las siguientes transformaciones:
1. **Extraer**: Leer el dataset limpio.
2. **Transformar**: Calcular métricas agregadas por país y por producto.
3. **Cargar**: Guardar los resultados en una base de datos SQLite y en archivos CSV.

Mediremos los tiempos de cada etapa para entender el rendimiento.

### **3.1. Pipeline Batch: Agregación por País**

Calcularemos:
*   Total de ventas por país.
*   Número de transacciones por país.
*   Ticket promedio por país.

In [ ]:
print("--- PIPELINE BATCH: AGREGACIÓN POR PAÍS ---")

# Medir tiempo total del pipeline
pipeline_start = time.time()

# --- EXTRACT: Datos ya están en df_clean ---
extract_start = time.time()
# (Simulamos que extraemos de una fuente)
extract_time = time.time() - extract_start
print(f"\n📤 EXTRACT completado en {extract_time:.4f} segundos")

# --- TRANSFORM: Agregaciones por país ---
transform_start = time.time()

# Agrupar por país
country_stats = df_clean.groupby('Country').agg({
    'InvoiceNo': 'count',        # Número de transacciones
    'TotalAmount': 'sum',        # Ventas totales
    'CustomerID': 'nunique'      # Clientes únicos
}).rename(columns={
    'InvoiceNo': 'num_transacciones',
    'TotalAmount': 'ventas_totales',
    'CustomerID': 'clientes_unicos'
}).reset_index()

# Calcular ticket promedio
country_stats['ticket_promedio'] = country_stats['ventas_totales'] / country_stats['num_transacciones']

# Ordenar por ventas totales descendente
country_stats = country_stats.sort_values('ventas_totales', ascending=False).reset_index(drop=True)

transform_time = time.time() - transform_start
print(f"🔄 TRANSFORM completado en {transform_time:.4f} segundos")

# Mostrar resultados
print("\nTop 10 países por ventas totales:")
display(country_stats.head(10))

# --- LOAD: Guardar resultados ---
load_start = time.time()

# Guardar a CSV
country_stats.to_csv('ventas_por_pais.csv', index=False)
print("  ✅ Guardado a 'ventas_por_pais.csv'")

# Guardar a SQLite
conn = sqlite3.connect('resultados_batch.db')
country_stats.to_sql('ventas_por_pais', conn, if_exists='replace', index=False)
print("  ✅ Guardado a SQLite (tabla 'ventas_por_pais')")

load_time = time.time() - load_start
pipeline_time = time.time() - pipeline_start

print(f"\n📥 LOAD completado en {load_time:.4f} segundos")
print(f"\n⏱️  TIEMPO TOTAL DEL PIPELINE: {pipeline_time:.4f} segundos")

# Visualización
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
top10 = country_stats.head(10)
plt.barh(top10['Country'], top10['ventas_totales'], color='skyblue')
plt.xlabel('Ventas Totales')
plt.title('Top 10 Países por Ventas Totales')
plt.gca().invert_yaxis()

plt.subplot(1, 2, 2)
plt.barh(top10['Country'], top10['ticket_promedio'], color='lightcoral')
plt.xlabel('Ticket Promedio')
plt.title('Ticket Promedio por País (Top 10)')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

### **3.2. Pipeline Batch: Agregación por Producto**

Ahora procesaremos por producto para identificar los más vendidos.

In [ ]:
print("--- PIPELINE BATCH: AGREGACIÓN POR PRODUCTO ---")

pipeline_start = time.time()

# --- EXTRACT ---
extract_start = time.time()
# Datos ya en df_clean
extract_time = time.time() - extract_start

# --- TRANSFORM ---
transform_start = time.time()

# Agrupar por producto (código y descripción)
product_stats = df_clean.groupby(['StockCode', 'Description']).agg({
    'Quantity': 'sum',           # Unidades totales vendidas
    'TotalAmount': 'sum',        # Ingresos totales
    'InvoiceNo': 'count',        # Número de transacciones
    'Country': 'nunique'         # Países donde se vendió
}).rename(columns={
    'Quantity': 'unidades_vendidas',
    'TotalAmount': 'ingresos_totales',
    'InvoiceNo': 'num_transacciones',
    'Country': 'paises_distintos'
}).reset_index()

# Calcular precio promedio ponderado
product_stats['precio_promedio'] = product_stats['ingresos_totales'] / product_stats['unidades_vendidas']

# Ordenar por ingresos
product_stats = product_stats.sort_values('ingresos_totales', ascending=False).reset_index(drop=True)

transform_time = time.time() - transform_start

# --- LOAD ---
load_start = time.time()

# Guardar a CSV
product_stats.head(100).to_csv('top_productos.csv', index=False)
print("  ✅ Guardado top 100 productos a 'top_productos.csv'")

# Guardar a SQLite
product_stats.to_sql('ventas_por_producto', conn, if_exists='replace', index=False)
print("  ✅ Guardado completo a SQLite (tabla 'ventas_por_producto')")

load_time = time.time() - load_start
pipeline_time = time.time() - pipeline_start

print(f"\n📤 EXTRACT: {extract_time:.4f}s")
print(f"🔄 TRANSFORM: {transform_time:.4f}s")
print(f"📥 LOAD: {load_time:.4f}s")
print(f"\n⏱️  TIEMPO TOTAL: {pipeline_time:.4f} segundos")

# Mostrar top 10 productos
print("\nTop 10 productos por ingresos:")
display(product_stats.head(10))

# Visualizar top productos
plt.figure(figsize=(10, 6))
top10_prod = product_stats.head(10)
# Acortar descripciones para visualización
short_desc = [d[:30] + '...' if len(d) > 30 else d for d in top10_prod['Description']]
plt.barh(short_desc, top10_prod['ingresos_totales'], color='lightgreen')
plt.xlabel('Ingresos Totales')
plt.title('Top 10 Productos por Ingresos')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Cerrar conexión a BD
conn.close()

---
## **Parte 4: Medición y Comparación de Tiempos**

Ahora realizaremos un experimento para comparar el rendimiento de diferentes enfoques al procesar el mismo conjunto de datos. Esto es crucial para optimizar pipelines batch en producción.

In [ ]:
print("--- EXPERIMENTO: COMPARACIÓN DE RENDIMIENTO ---")

# Crear subsets de diferentes tamaños para probar escalabilidad
sizes = [10000, 50000, 100000, 200000, 500000]
sizes = [s for s in sizes if s <= len(df_clean)]  # Ajustar al tamaño real

results = []

for size in sizes:
    print(f"\n📊 Probando con {size} registros...")
    
    # Tomar muestra
    df_sample = df_clean.head(size)
    
    # --- Método 1: pandas estándar (vectorizado) ---
    start = time.time()
    result_pandas = df_sample.groupby('Country').agg({
        'TotalAmount': 'sum',
        'InvoiceNo': 'count'
    }).reset_index()
    pandas_time = time.time() - start
    
    # --- Método 2: iteración manual (ineficiente) ---
    start = time.time()
    countries = {}
    for _, row in df_sample.iterrows():
        country = row['Country']
        if country not in countries:
            countries[country] = {'TotalAmount': 0, 'InvoiceNo': 0}
        countries[country]['TotalAmount'] += row['TotalAmount']
        countries[country]['InvoiceNo'] += 1
    iter_time = time.time() - start
    
    results.append({
        'size': size,
        'pandas_time': pandas_time,
        'iteration_time': iter_time,
        'speedup': iter_time / pandas_time if pandas_time > 0 else 0
    })
    
    print(f"    pandas: {pandas_time:.4f}s, iteración: {iter_time:.4f}s (x{iter_time/pandas_time:.1f} más lento)")

# Crear DataFrame con resultados
df_results = pd.DataFrame(results)
print("\n--- RESULTADOS COMPARATIVOS ---")
display(df_results)

# Visualizar comparación
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(df_results['size'], df_results['pandas_time'], 'o-', label='pandas (vectorizado)', linewidth=2)
plt.plot(df_results['size'], df_results['iteration_time'], 's-', label='iteración manual', linewidth=2)
plt.xlabel('Número de registros')
plt.ylabel('Tiempo de ejecución (segundos)')
plt.title('Comparación de Rendimiento')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(df_results['size'], df_results['speedup'], 'ro-', linewidth=2)
plt.xlabel('Número de registros')
plt.ylabel('Factor de desaceleración (iteración/pandas)')
plt.title('Iteración manual vs pandas (más alto = peor)')
plt.grid(True)
plt.axhline(y=1, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("\n📌 Conclusión: Las operaciones vectorizadas de pandas son órdenes de magnitud más eficientes que la iteración manual, especialmente a medida que crece el volumen de datos.")

---
## **Parte 5: Optimización - Procesamiento por Fragmentos (Chunking)**

Cuando los datasets son demasiado grandes para caber en memoria, una técnica común es procesarlos por fragmentos (chunks). Esto es especialmente útil en pipelines batch con limitaciones de recursos.

In [ ]:
print("--- PROCESAMIENTO POR FRAGMENTOS (CHUNKING) ---")

# Simularemos un archivo grande procesándolo en fragmentos
# Primero, guardamos el dataset limpio como CSV para leerlo en fragmentos
df_clean.to_csv('dataset_completo.csv', index=False)
print("Dataset guardado como 'dataset_completo.csv'")

# Configuración de fragmentos
chunk_size = 50000  # Procesar de 50k en 50k registros
total_chunks = 0
chunk_times = []

# Acumuladores para agregación por país
country_aggregator = {}

print(f"\nProcesando por fragmentos de {chunk_size} registros...")

start_total = time.time()

# Leer CSV en fragmentos
for i, chunk in enumerate(pd.read_csv('dataset_completo.csv', chunksize=chunk_size)):
    chunk_start = time.time()
    
    # Procesar cada fragmento
    for _, row in chunk.iterrows():
        country = row['Country']
        amount = row['TotalAmount']
        
        if country not in country_aggregator:
            country_aggregator[country] = {'TotalAmount': 0, 'Count': 0}
        country_aggregator[country]['TotalAmount'] += amount
        country_aggregator[country]['Count'] += 1
    
    chunk_time = time.time() - chunk_start
    chunk_times.append(chunk_time)
    total_chunks += 1
    print(f"  Fragmento {i+1}: {chunk_time:.2f}s")

total_time = time.time() - start_total

# Convertir acumuladores a DataFrame
df_chunk_result = pd.DataFrame.from_dict(country_aggregator, orient='index').reset_index()
df_chunk_result.columns = ['Country', 'TotalAmount', 'Count']
df_chunk_result = df_chunk_result.sort_values('TotalAmount', ascending=False).reset_index(drop=True)

print(f"\n✅ Procesamiento por fragmentos completado.")
print(f"   Fragmentos procesados: {total_chunks}")
print(f"   Tiempo promedio por fragmento: {np.mean(chunk_times):.2f}s")
print(f"   Tiempo total: {total_time:.2f}s")

print("\nResultados del procesamiento por fragmentos (Top 10 países):")
display(df_chunk_result.head(10))

# Comparar con procesamiento en memoria
print("\n--- COMPARACIÓN CON PROCESAMIENTO EN MEMORIA ---")
start_mem = time.time()
mem_result = df_clean.groupby('Country')['TotalAmount'].agg(['sum', 'count']).reset_index()
mem_time = time.time() - start_mem
print(f"Procesamiento en memoria: {mem_time:.2f}s")
print(f"Procesamiento por fragmentos: {total_time:.2f}s")
print(f"Relación (fragmentos/memoria): {total_time/mem_time:.2f}x")

print("\n📌 El procesamiento por fragmentos es más lento pero permite trabajar con datasets que no caben en memoria. En producción, se usa cuando la memoria es limitada.")

---
## **Ejercicios Prácticos para el Estudiante**

Ahora aplica lo aprendido con estos ejercicios basados en el dataset real.

### **Ejercicio 1: Pipeline batch mensual**

Crea un pipeline batch que:
1.  Extraiga el dataset completo.
2.  Transforme: calcule las ventas totales por **mes** (extrae el mes de `InvoiceDate`).
3.  Cargue el resultado en una tabla `ventas_mensuales` en SQLite.

Muestra los resultados ordenados por mes. Mide el tiempo total del pipeline.

In [ ]:
# --- INICIO DE TU CÓDIGO ---

# Asegurar que InvoiceDate es datetime
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# Crear columna de mes (formato YYYY-MM)
df_clean['Mes'] = df_clean['InvoiceDate'].dt.to_period('M')

# Pipeline
start_time = time.time()

# Transform: Agrupar por mes
ventas_mensuales = df_clean.groupby('Mes').agg({
    'TotalAmount': 'sum',
    'InvoiceNo': 'count',
    'CustomerID': 'nunique'
}).rename(columns={
    'TotalAmount': 'ventas_totales',
    'InvoiceNo': 'num_transacciones',
    'CustomerID': 'clientes_unicos'
}).reset_index()

# Convertir Mes a string para guardar
ventas_mensuales['Mes'] = ventas_mensuales['Mes'].astype(str)

# Load a SQLite
conn = sqlite3.connect('resultados_batch.db')
ventas_mensuales.to_sql('ventas_mensuales', conn, if_exists='replace', index=False)
conn.close()

elapsed_time = time.time() - start_time

print(f"Tiempo total del pipeline: {elapsed_time:.2f} segundos")
print("\nVentas mensuales:")
display(ventas_mensuales.sort_values('Mes'))

# Visualización
plt.figure(figsize=(12, 5))
plt.plot(ventas_mensuales['Mes'], ventas_mensuales['ventas_totales'], 'o-', linewidth=2)
plt.xlabel('Mes')
plt.ylabel('Ventas Totales')
plt.title('Ventas Mensuales')
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 2: Análisis de clientes RFM simplificado**

Implementa un pipeline batch que calcule para cada cliente:
*   **Recencia**: Días desde la última compra (usar la fecha máxima del dataset como "hoy").
*   **Frecuencia**: Número de transacciones.
*   **Monetario**: Total gastado.

Guarda los resultados en `rfm_clientes.csv`. Mide el tiempo de ejecución.

In [ ]:
# --- INICIO DE TU CÓDIGO ---

# Asegurar que InvoiceDate es datetime
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# Fecha máxima como "hoy"
hoy = df_clean['InvoiceDate'].max()
print(f"Fecha de referencia (hoy): {hoy}")

# Pipeline
start_time = time.time()

# Calcular métricas por cliente
rfm = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (hoy - x.max()).days,  # Recencia (días desde última compra)
    'InvoiceNo': 'count',                            # Frecuencia
    'TotalAmount': 'sum'                             # Monetario
}).rename(columns={
    'InvoiceDate': 'Recencia',
    'InvoiceNo': 'Frecuencia',
    'TotalAmount': 'Monetario'
}).reset_index()

# Ordenar por monetario descendente
rfm = rfm.sort_values('Monetario', ascending=False).reset_index(drop=True)

# Guardar a CSV
rfm.to_csv('rfm_clientes.csv', index=False)

elapsed_time = time.time() - start_time

print(f"Tiempo de ejecución: {elapsed_time:.2f} segundos")
print("\nTop 10 clientes por valor monetario:")
display(rfm.head(10))

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(rfm['Recencia'], bins=30, color='skyblue', edgecolor='black')
axes[0].set_xlabel('Recencia (días)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de Recencia')

axes[1].hist(rfm['Frecuencia'], bins=30, color='lightgreen', edgecolor='black')
axes[1].set_xlabel('Frecuencia (transacciones)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de Frecuencia')

axes[2].hist(rfm['Monetario'], bins=30, color='lightcoral', edgecolor='black')
axes[2].set_xlabel('Monetario (total gastado)')
axes[2].set_ylabel('Frecuencia')
axes[2].set_title('Distribución de Monetario')

plt.tight_layout()
plt.show()

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 3: Comparación de rendimiento con diferentes tamaños de chunk**

Modifica el código de la Parte 5 para probar diferentes tamaños de chunk (por ejemplo: 10000, 50000, 100000, 200000) y crea una gráfica que muestre cómo varía el tiempo total de procesamiento con el tamaño del chunk.

¿Qué tamaño de chunk parece óptimo para este dataset?

In [ ]:
# --- INICIO DE TU CÓDIGO ---

chunk_sizes = [10000, 50000, 100000, 200000]
chunk_results = []

print("--- EXPERIMENTO: VARIACIÓN DEL TAMAÑO DE CHUNK ---")

for chunk_size in chunk_sizes:
    print(f"\nProbando chunk_size = {chunk_size}...")
    
    # Acumuladores
    country_agg = {}
    chunk_times = []
    
    start_total = time.time()
    num_chunks = 0
    
    # Procesar por chunks
    for i, chunk in enumerate(pd.read_csv('dataset_completo.csv', chunksize=chunk_size)):
        chunk_start = time.time()
        
        # Procesar chunk
        for _, row in chunk.iterrows():
            country = row['Country']
            amount = row['TotalAmount']
            
            if country not in country_agg:
                country_agg[country] = {'TotalAmount': 0, 'Count': 0}
            country_agg[country]['TotalAmount'] += amount
            country_agg[country]['Count'] += 1
        
        chunk_time = time.time() - chunk_start
        chunk_times.append(chunk_time)
        num_chunks += 1
    
    total_time = time.time() - start_total
    
    chunk_results.append({
        'chunk_size': chunk_size,
        'total_time': total_time,
        'num_chunks': num_chunks,
        'avg_chunk_time': np.mean(chunk_times),
        'total_records': num_chunks * chunk_size
    })
    
    print(f"  Tiempo total: {total_time:.2f}s, Chunks: {num_chunks}, Tiempo promedio por chunk: {np.mean(chunk_times):.2f}s")

# Crear DataFrame con resultados
df_chunk_exp = pd.DataFrame(chunk_results)
print("\n--- RESULTADOS DEL EXPERIMENTO ---")
display(df_chunk_exp)

# Visualizar
plt.figure(figsize=(10, 5))
plt.plot(df_chunk_exp['chunk_size'], df_chunk_exp['total_time'], 'o-', linewidth=2, markersize=8)
plt.xlabel('Tamaño de chunk')
plt.ylabel('Tiempo total de procesamiento (s)')
plt.title('Tiempo de procesamiento vs Tamaño de chunk')
plt.grid(True)
plt.show()

# Identificar el chunk_size óptimo
optimo = df_chunk_exp.loc[df_chunk_exp['total_time'].idxmin()]
print(f"\n📌 Tamaño de chunk óptimo: {optimo['chunk_size']} (tiempo: {optimo['total_time']:.2f}s)")

# --- FIN DE TU CÓDIGO ---

---
## **Conclusiones de la Semana 9**

En este notebook hemos trabajado con un dataset real y hemos:

✅ **Implementado pipelines batch completos** con extracción, transformación y carga.

✅ **Medido tiempos de ejecución** para entender el rendimiento de diferentes operaciones.

✅ **Comparado enfoques** (vectorizado vs iteración manual) y observado la enorme diferencia de eficiencia.

✅ **Aplicado procesamiento por fragmentos** (chunking) para manejar datasets que no caben en memoria.

✅ **Experimentado con diferentes tamaños de chunk** para encontrar la configuración óptima.

Estas técnicas son fundamentales para construir pipelines de datos robustos y eficientes en entornos de producción.

---
**Fin del Notebook - Semana 9**